In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOStateDecider
# from nocode_robot_programming.state_decision.dino_model_v2 import DINOFeaturePresence
from nocode_robot_programming.state_decision.dino_model_v3 import DINOFeaturePresence
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
from torch.utils.data import DataLoader
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag
from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing
from nocode_robot_programming.jupyter_plot import jupyter_plot as inline_plot, show_gray_video_cuda

from copy import deepcopy
seed = 50
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

dataset = TrajectoryDataset(trajectory_data.package_path)

In [ ]:
def option2_trials_are_test_data(dataset, file_names: list[str], tags: list[str] = [], at: slice = slice(35,85)) -> ImageDatasetView:
    X = torch.tensor([])
    Xt = torch.tensor([])
    y_int = torch.tensor([], dtype=torch.int)
    y_names = []

    # tags = []
    # for file in file_names:
    #     t = dataset[file]['tag']
    #     if t not in tags:
    #         tags.append(t)

    X_parts, Xt_parts, y_int_parts, y_name_parts = [], [], [], []

    start = at.start if at.start is not None else -float("inf")
    stop  = at.stop  if at.stop  is not None else  float("inf")

    for file in file_names:
        f = Filename(file)
        idx = dataset.files.index(f"{dataset.dir}/{file}.npz")
        imgs = saved_img_processing(dataset[idx]['img'].squeeze()).squeeze()   # (nsamples, H, W)
        nsamples = imgs.shape[0]
        
        tag = dataset[file]['tag']
        i = tags.index(tag)
        xt = torch.arange(f.offset, f.offset + nsamples)  # shape (nsamples,)
        
        mask = (xt >= start) & (xt < stop)                # safe, no OOB
        if mask.any():
            imgs_sub = imgs[mask]
            xt_sub   = xt[mask]

            y_int_sub   = torch.full((imgs_sub.shape[0],), i, dtype=torch.int)
            y_names_sub = [tag] * imgs_sub.shape[0]

            X_parts.append(imgs_sub)
            Xt_parts.append(xt_sub)
            y_int_parts.append(y_int_sub)
            y_name_parts.extend(y_names_sub)

    X  = torch.cat(X_parts, dim=0)             # (total_samples, H, W)
    Xt = torch.cat(Xt_parts, dim=0)            # (total_samples,)
    y_int = torch.cat(y_int_parts, dim=0)      # (total_samples,)
    y_names = y_name_parts

    return ImageDatasetView(X=X, Xt=Xt, y_int=y_int, y_names=y_names, y_cls=tags)

In [ ]:
def custom_datasets():
    datasets = []
    
    task_name = 'd1_peg_pick'
    window = 10
    branch_offset = 49
    d_test = option2_trials_are_test_data(dataset, tags=['d1_peg_pick', 'd1_peg_pick_branch_at_49'],
        file_names=[n for n in dataset.tasks[task_name]['names'] if "trial" not in n or n == 'd1_peg_pick_trial_3'],
        at=slice(branch_offset-window/2.0, branch_offset+window/2.0))
    d_train = option2_trials_are_test_data(dataset, tags=['d1_peg_pick', 'd1_peg_pick_branch_at_49'],
        file_names=[n for n in dataset.tasks[task_name]['names'] if "trial" in n and n != 'd1_peg_pick_trial_3'],
        at=slice(branch_offset-window/2, branch_offset+window/2))
    print("Train set: ", [n for n in dataset.tasks[task_name]['names'] if "trial" not in n or n == 'd1_peg_pick_trial_3'],
        "\nTest set: ", [n for n in dataset.tasks[task_name]['names'] if "trial" in n and n != 'd1_peg_pick_trial_3'])
    
    datasets.append([d_train, d_test])
    datasets.append([d_test, d_train])

    names_and_tags = [f"{name}: {tag}" for name,tag in zip(dataset.tasks[task_name]['names'], dataset.tasks[task_name]['tags'])]
    print(names_and_tags)

    task_name = 'd1_peg_pick'
    window = 10
    d2_train = option2_trials_are_test_data(dataset, tags=['d1_peg_pick', 'd1_peg_pick_branch_at_49'],
        file_names=['d1_peg_pick', 'd1_peg_pick_branch_at_49', 'd1_peg_pick_trial_0', 'd1_peg_pick_trial_4', 'd1_peg_pick_trial_5', 'd1_peg_pick_trial_1'],
        at=slice(49-window/2, 49+window/2))

    d2_test = option2_trials_are_test_data(dataset, tags=['d1_peg_pick', 'd1_peg_pick_branch_at_49'],
        file_names=['d1_peg_pick_trial_2', 'd1_peg_pick_trial_3', 'd1_peg_pick_trial_7', 'd1_peg_pick_trial_8', 'd1_peg_pick_trial_6'],
        at=slice(49-window/2, 49+window/2))
    
    datasets.append([d2_train, d2_test])

    d_train_dupl = deepcopy(d_train)
    d_train_dupl.X = torch.vstack([d_train.X] * 20)
    d_train_dupl.y_int = torch.vstack([d_train.y_int] * 20)
    d_train_dupl.y_names = [*d_train.y_names] * 20

    datasets.append([d_train_dupl, d2_test])

    return datasets

In [ ]:
ds1, ds2, ds3, ds4 = custom_datasets()
d_train, d_test = ds4

In [ ]:
d_train.play_video(fps=10)
d_test.play_video(fps=10)

In [ ]:
display(show_gray_video_cuda(d_train.X, fps=20))

In [ ]:
d_train.X.shape

In [ ]:
# for task_name in dataset.tasks.keys():
for decider in [
        # DINOFeaturePresence(percentile_keep=0.0),
        # DINOStateDecider(dino_variant = "dinov2_vitl14", percent_keep=0),
        # StateDeciderSIFT(),
        AEGP(),
    ]:
    decider.train(d_train.X, d_train.y_int, d_train.y_cls)
    inline_plot.plt_save() # loss fig obtained in train function

    y_pred = decider.predict_many(d_train.X)
    
    pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False)
    inline_plot.plt_save()  # confustion matrix

    y_pred = decider.predict_many(d_test.X)
    pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False)
    inline_plot.plt_save()  # confustion matrix

    inline_plot.show()

In [ ]:
reconstructed_images = decider.videoembedder.model.forward(d_train.X[23:34].unsqueeze(1)).squeeze()
display(show_gray_video_cuda(torch.hstack([d_train.X[23:34], reconstructed_images]), fps=20))

In [ ]:
y_pred = decider.predict_many(d_test.X)
pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False)
inline_plot.plt_save()  # confustion matrix

inline_plot.show()